# Hybrid LLM-BO Agent

This notebook demonstrates BO-08 with a deterministic fake LLM by default. The LLM-like layer inspects history, requests BO candidates, reviews candidate IDs, and decides when to continue or stop. BO remains the numerical optimizer and the backend executes only validated candidate IDs.

The virtual laboratory is educational. It is useful for learning agent orchestration and safety boundaries, not for predicting real HfO2/MoS2 chemistry.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from asd_agent.bo.hybrid_agent import (  # noqa: E402
    FakeHybridLLM,
    HybridLLMBOAgent,
    LiteratureHit,
    MockLiteratureProvider,
    hybrid_tool_schemas,
)
from asd_agent.bo.stage2_mobo import Stage2BOSettings  # noqa: E402
from asd_agent.config import load_stage2_scenario  # noqa: E402

config = load_stage2_scenario("inherent_selectivity")
settings = Stage2BOSettings(
    experiment_budget=3,
    initial_design_size=2,
    qmc_samples=8,
    num_restarts=1,
    raw_samples=8,
    acquisition_timeout_s=2.0,
    candidate_cycle_values=[30, 50, 70],
    random_fallback_points=32,
)
len(hybrid_tool_schemas())

## Run With The Fake LLM

The fake LLM exercises inspection, local literature lookup, soft-bound review, BO candidate generation, candidate-ID execution, observation, and stopping. No live API call is made.

In [ ]:
literature = MockLiteratureProvider(
    {
        "area": [
            LiteratureHit(
                source_id="mock_asd_note",
                title="Mock ASD optimization note",
                summary="A local educational note used by the fake LLM.",
            )
        ]
    }
)

agent = HybridLLMBOAgent(
    config,
    mode="hybrid_intervention",
    llm=FakeHybridLLM("intervention"),
    literature_provider=literature,
    bo_settings=settings,
    seed=123,
)
result = agent.run(budget=3)
result.status, len(result.observations), len(result.candidates), len(result.literature)

In [ ]:
[(event.from_state, event.tool_name, event.to_state, event.status) for event in result.events]

## Safety Boundary

`run_virtual_experiment` accepts `candidate_id`, not arbitrary reactor settings. Numerical conditions enter the backend only through BO-created candidate records.

In [ ]:
run_virtual_schema = {schema["name"]: schema for schema in hybrid_tool_schemas()}[
    "run_virtual_experiment"
]
run_virtual_schema["parameters"]["properties"].keys()